# PYDANTIC WAY

In [1]:
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
import os

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b") 

e:\Agents\agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie"),
    year:int=Field(description="The released year"),
    dir:str=Field(description="The director of the flim")
    rating:float=Field(description="The movies rating")

model_with_structure=model.with_structured_output(Movie)

model_with_structure.invoke("Provide details of the movie titanic!")

e:\Agents\agent\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\Agents\agent\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The released year'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


Movie(title='Titanic', year=1997, dir='James Cameron', rating=7.9)

In [5]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(...,description="The title of the movie"),
    year:int=Field(...,description="The released year"),
    dir:str=Field(...,description="The director of the flim")
    rating:float=Field(...,description="The movies rating")

model_with_structure=model.with_structured_output(Movie,include_raw=True)

model_with_structure.invoke("Provide details of the movie titanic!")
model_with_structure

e:\Agents\agent\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
e:\Agents\agent\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The released year'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


{
  raw: _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x0000016BDCAA3080>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016BDCA7A750>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': 'A movie with details', 'parameters': {'properties': {'title': {'type': 'string'}, 'year': {'type': 'integer'}, 'dir': {'description': 'The director of the flim', 'type': 'string'}, 'rating': {'description': 'The movies rating', 'type': 'number'}}, 'required': ['dir', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'description': 'A movie with details'

In [6]:
# nested strucutre
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float|None=Field(None,description="Budget in millions USD") #a budget value or nothing | -> OR
model_with_structure=model.with_structured_output(MovieDetails)

resp=model_with_structure.invoke("Provide details about the movie Inception")
resp

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal Cobb')], genres=['Action', 'Sci-Fi', 'Thriller', 'Adventure'], budget=160.0)

# TYPEDICT WAY

In [8]:
from typing_extensions import TypedDict,Annotated

class Moviedict(BaseModel):
    """A movie with details"""
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"The released year"]
    dir:Annotated[str,...,"The director of the flim"]
    rating:Annotated[float,...,"The movies rating"]

model_with_structure=model.with_structured_output(Moviedict)
model_with_structure.invoke("Provide detials of avengers endgame")


Moviedict(title='Avengers: Endgame', year=2019, dir='Anthony and Joe Russo', rating=8.4)

In [11]:
# nested structure same like pydantic
from typing_extensions import TypedDict

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None

model_with_typedict = model.with_structured_output(MovieDetails)

resp = model_with_typedict.invoke("Provide details about the movie avenger end game")
resp

{'budget': 356000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Josh Brolin', 'role': 'Thanos'}],
 'genres': ['Action', 'Adventure', 'Science Fiction', 'Drama'],
 'title': 'Avengers: Endgame',
 'year': 2019}

# DATACLASSES WAY

In [12]:
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
import os

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b") 

In [ ]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name:str=Field(description="The name of the person")
    email:str=Field(description="The email address of the person")
    phone:str=Field(description="The phone number of the person")

agent=create_agent(
    model=model,
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from john doe, john@example.com , (555), 123-4567"}]
})

print(result["structured_response"]) #-> ["structured_response"] used this to have a structured output what u mentioned inaside the base model!

name='john doe' email='john@example.com' phone='(555) 123-4567'
